<a href="https://colab.research.google.com/github/vedxantt/Vedant-Mishra-2026B15410080/blob/main/food_delivery_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 8: Food Delivery Operations Analytics
Tools: pandas + numpy only (no plotting libraries). Upload `08_food_delivery.csv` to Colab first.

In [39]:
# Setup
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# If the file is not in the Colab session yet, uncomment these two lines and upload it:
# from google.colab import files
# files.upload()

## Task 1: Load and inspect the order dataset

In [40]:
# Task 1
df = pd.read_csv("08_food_delivery.csv")
print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nData types:\n", df.dtypes)
print("\nFirst 5 rows:")
display(df.head())
print("\nSummary statistics:")
display(df.describe(include="all").T)

Shape: (1525, 10)

Columns: ['Order_ID', 'Order_Date', 'Order_Hour', 'City', 'Restaurant', 'Cuisine', 'Order_Value', 'Payment_Mode', 'Order_Status', 'Delivery_Time_Min']

Data types:
 Order_ID              object
Order_Date            object
Order_Hour             int64
City                  object
Restaurant            object
Cuisine               object
Order_Value          float64
Payment_Mode          object
Order_Status          object
Delivery_Time_Min    float64
dtype: object

First 5 rows:


,Order_ID,Order_Date,Order_Hour,City,Restaurant,Cuisine,Order_Value,Payment_Mode,Order_Status,Delivery_Time_Min
0,FD10494,2026-10-26,19,Mumbai,Healthy Bowl,Healthy,274.00,UPI,Delivered,32.6
1,FD10017,2026-10-22,12,Pune,Spice Route,North Indian,226.79,UPI,Delivered,27.1
2,FD10289,2026-10-14,20,Mumbai,Tandoori Nights,North Indian,550.24,Cash,Delayed,58.1
3,FD11005,2026-09-14,18,Kolkata,Spice Route,North Indian,338.32,Cash,Delayed,104.7
4,FD11238,2026-10-28,20,Mumbai,Burger Barn,Burgers,118.48,Cash,Delivered,43.6



Summary statistics:


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Order_ID,1525,1500,FD11286,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order_Date,1525,92,2026-09-17,30,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order_Hour,1525.0,NaN,NaN,NaN,17.04918,3.701751,10.0,13.0,18.0,20.0,23.0
City,1525,8,Delhi,310,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Restaurant,1525,12,Sweet Tooth,142,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Cuisine,1525,9,North Indian,266,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order_Value,1525.0,NaN,NaN,NaN,305.409725,117.958408,80.0,221.16,300.3,382.58,681.17
Payment_Mode,1525,4,UPI,710,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Order_Status,1525,3,Delivered,1103,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Delivery_Time_Min,1394.0,NaN,NaN,NaN,38.85,12.221836,12.0,31.5,37.8,44.1,109.5


## Task 2: Missing delivery times

In [41]:
# Task 2
print("Missing values per column:\n", df.isna().sum())
miss = df[df["Delivery_Time_Min"].isna()]
print("\nMissing delivery times by order status:")
print(miss["Order_Status"].value_counts())

# Handling decision:
# - Cancelled orders were never delivered, so a missing time is expected -> keep as NaN (excluded from averages)
# - Other missing values (Delivered/Delayed) are data gaps -> fill with the median delivery time of the same city
mask = df["Delivery_Time_Min"].isna() & (df["Order_Status"] != "Cancelled")
city_median = df.groupby("City")["Delivery_Time_Min"].transform("median")
df.loc[mask, "Delivery_Time_Min"] = city_median[mask]
print("\nRemaining missing delivery times (all should be Cancelled):", df["Delivery_Time_Min"].isna().sum())
print("Of which cancelled:", (df["Delivery_Time_Min"].isna() & (df["Order_Status"] == "Cancelled")).sum())

Missing values per column:
 Order_ID               0
Order_Date             0
Order_Hour             0
City                   0
Restaurant             0
Cuisine                0
Order_Value            0
Payment_Mode           0
Order_Status           0
Delivery_Time_Min    131
dtype: int64

Missing delivery times by order status:
Order_Status
Cancelled    107
Delivered     22
Delayed        2
Name: count, dtype: int64

Remaining missing delivery times (all should be Cancelled): 107
Of which cancelled: 107


## Task 3: Duplicate Order_ID values

In [42]:
# Task 3
dup_mask = df["Order_ID"].duplicated(keep=False)
print("Rows involved in duplicates:", dup_mask.sum())
print("Extra duplicate rows:", df["Order_ID"].duplicated().sum())
display(df[dup_mask].sort_values("Order_ID").head(10))
df = df.drop_duplicates(subset="Order_ID", keep="first").reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)

Rows involved in duplicates: 50
Extra duplicate rows: 25


,Order_ID,Order_Date,Order_Hour,City,Restaurant,Cuisine,Order_Value,Payment_Mode,Order_Status,Delivery_Time_Min
1450,FD10076,2026-09-13,23,Bengaluru,Curry Leaf,South Indian,233.08,Cash,Delivered,32.0
636,FD10076,2026-09-13,23,Bengaluru,Curry Leaf,South Indian,233.08,Cash,Delivered,32.0
224,FD10092,2026-09-17,20,Mumbai,Dragon Wok,Chinese,99.44,Cash,Delayed,56.2
488,FD10092,2026-09-17,20,Mumbai,Dragon Wok,Chinese,99.44,Cash,Delayed,56.2
729,FD10102,2026-10-21,20,Meerut,Spice Route,North Indian,305.05,Card,Delivered,32.5
332,FD10102,2026-10-21,20,Meerut,Spice Route,North Indian,305.05,Card,Delivered,32.5
808,FD10103,2026-09-14,22,Meerut,Biryani House,Biryani,407.87,Card,Cancelled,NaN
867,FD10103,2026-09-14,22,Meerut,Biryani House,Biryani,407.87,Card,Cancelled,NaN
499,FD10170,2026-09-07,20,Meerut,Wrap Station,Fast Food,173.45,UPI,Delivered,40.8
284,FD10170,2026-09-07,20,Meerut,Wrap Station,Fast Food,173.45,UPI,Delivered,40.8


Shape after removing duplicates: (1500, 10)


## Task 4: Total orders and order-status percentages

In [43]:
# Task 4
total_orders = len(df)
print("Total orders:", total_orders)
status_pct = (df["Order_Status"].value_counts(normalize=True) * 100).round(2)
display(status_pct.rename("Percentage (%)").to_frame())

Total orders: 1500


,Percentage (%)
Order_Status,
Delivered,72.47
Delayed,20.67
Cancelled,6.87


## Task 5: Average delivery time by city

In [44]:
# Task 5
avg_city = df.groupby("City")["Delivery_Time_Min"].mean().round(2).sort_values()
display(avg_city.rename("Avg Delivery Time (min)").to_frame())

,Avg Delivery Time (min)
City,
Meerut,30.20
Pune,34.56
Chennai,36.42
Hyderabad,36.83
Bengaluru,38.08
Kolkata,40.60
Delhi,41.18
Mumbai,43.60


## Task 6: Top 5 restaurants by total order value

In [45]:
# Task 6
top5 = (df.groupby("Restaurant")["Order_Value"].sum().round(2)
          .sort_values(ascending=False).head(5))
display(top5.rename("Total Order Value").to_frame())

,Total Order Value
Restaurant,
Pizza Corner,57511.11
Tandoori Nights,53427.16
Biryani House,45391.08
Spice Route,41819.25
Noodle Bar,40576.85


## Task 7: Average order value by restaurant

In [46]:
# Task 7
avg_rest = (df.groupby("Restaurant")["Order_Value"].mean().round(2)
              .sort_values(ascending=False))
display(avg_rest.rename("Avg Order Value").to_frame())

,Avg Order Value
Restaurant,
Pizza Corner,463.80
Tandoori Nights,414.16
Biryani House,378.26
Dragon Wok,318.35
Spice Route,316.81
Noodle Bar,312.13
Healthy Bowl,305.45
Curry Leaf,268.13
Burger Barn,254.92


## Task 8: Peak ordering hours

In [47]:
# Task 8
hourly = df.groupby("Order_Hour").size().rename("Orders").to_frame()
hourly["Share (%)"] = (hourly["Orders"] / hourly["Orders"].sum() * 100).round(2)
print("Orders by hour:")
display(hourly)
print("Top 3 peak hours:")
peak_hours = hourly.sort_values("Orders", ascending=False).head(3)
display(peak_hours)

Orders by hour:


,Orders,Share (%)
Order_Hour,,
10,37,2.47
11,100,6.67
12,171,11.40
13,108,7.20
14,55,3.67
15,36,2.40
16,39,2.60
17,87,5.80
18,167,11.13


Top 3 peak hours:


,Orders,Share (%)
Order_Hour,,
19,232,15.47
20,225,15.00
12,171,11.40


## Task 9: Delivery time across order-status categories

In [48]:
# Task 9
status_time = df.groupby("Order_Status")["Delivery_Time_Min"].agg(["count", "mean", "median", "min", "max"]).round(2)
display(status_time)

,count,mean,median,min,max
Order_Status,,,,,
Cancelled,0,NaN,NaN,NaN,NaN
Delayed,310,54.83,50.25,37.2,109.5
Delivered,1087,34.27,35.30,12.0,45.0


## Task 10: Cancellation and delay rates by city

In [49]:
# Task 10
rates = pd.crosstab(df["City"], df["Order_Status"], normalize="index") * 100
city_rates = rates[["Cancelled", "Delayed"]].round(2)
city_rates.columns = ["Cancellation Rate (%)", "Delay Rate (%)"]
city_rates = city_rates.sort_values("Delay Rate (%)", ascending=False)
display(city_rates)

,Cancellation Rate (%),Delay Rate (%)
City,,
Mumbai,6.78,36.95
Delhi,6.51,22.80
Kolkata,7.62,20.95
Bengaluru,7.11,18.58
Hyderabad,8.67,15.61
Chennai,5.67,14.18
Pune,4.31,9.48
Meerut,8.18,3.64


## Task 11: NumPy median and standard deviation of delivery time

In [50]:
# Task 11
times = df["Delivery_Time_Min"].dropna().to_numpy()
print("Median delivery time (NumPy):", round(float(np.median(times)), 2), "min")
print("Std deviation (NumPy):", round(float(np.std(times, ddof=1)), 2), "min")
print("Mean for reference:", round(float(np.mean(times)), 2), "min")

Median delivery time (NumPy): 37.8 min
Std deviation (NumPy): 12.08 min
Mean for reference: 38.83 min


## Task 12: Unusually long deliveries using IQR

In [51]:
# Task 12
q1, q3 = np.percentile(times, [25, 75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1={q1:.2f}  Q3={q3:.2f}  IQR={iqr:.2f}  Upper fence={upper:.2f}")
outliers = df[df["Delivery_Time_Min"] > upper]
print("Unusually long deliveries:", len(outliers),
      f"({len(outliers) / df['Delivery_Time_Min'].notna().sum() * 100:.2f}% of delivered orders)")
display(outliers.sort_values("Delivery_Time_Min", ascending=False).head(10))
print("Outliers by city:")
display(outliers["City"].value_counts().rename("Outlier Orders").to_frame())

Q1=31.60  Q3=43.90  IQR=12.30  Upper fence=62.35
Unusually long deliveries: 42 (3.01% of delivered orders)


,Order_ID,Order_Date,Order_Hour,City,Restaurant,Cuisine,Order_Value,Payment_Mode,Order_Status,Delivery_Time_Min
1386,FD10063,2026-09-06,21,Delhi,Spice Route,North Indian,324.60,UPI,Delayed,109.5
3,FD11005,2026-09-14,18,Kolkata,Spice Route,North Indian,338.32,Cash,Delayed,104.7
1185,FD10276,2026-10-04,21,Hyderabad,Burger Barn,Burgers,240.76,Cash,Delayed,98.9
341,FD10340,2026-08-08,20,Delhi,Noodle Bar,Chinese,380.16,UPI,Delayed,94.3
33,FD11036,2026-09-22,11,Bengaluru,Sweet Tooth,Desserts,346.72,UPI,Delayed,93.8
1425,FD10685,2026-08-18,20,Bengaluru,Biryani House,Biryani,492.98,UPI,Delayed,93.4
261,FD10877,2026-10-23,18,Kolkata,Curry Leaf,South Indian,391.27,UPI,Delayed,92.8
1490,FD10918,2026-09-27,12,Mumbai,Burger Barn,Burgers,342.78,UPI,Delayed,92.7
667,FD10693,2026-10-07,22,Kolkata,Wrap Station,Fast Food,207.27,UPI,Delayed,92.7
624,FD10745,2026-09-22,20,Hyderabad,Wrap Station,Fast Food,89.10,UPI,Delayed,91.9


Outliers by city:


,Outlier Orders
City,
Delhi,11
Mumbai,9
Bengaluru,6
Kolkata,5
Hyderabad,5
Pune,3
Meerut,2
Chennai,1


## Task 13: Delivery-time bands (Fast / Standard / Delayed)

In [52]:
# Task 13
conditions = [
    df["Delivery_Time_Min"].isna(),
    df["Delivery_Time_Min"] <= 30,
    df["Delivery_Time_Min"] <= 45,
]
choices = ["No Delivery", "Fast", "Standard"]
df["Delivery_Band"] = np.select(conditions, choices, default="Delayed")

band_counts = df["Delivery_Band"].value_counts()
display(band_counts.rename("Orders").to_frame())
band_city = pd.crosstab(df["City"], df["Delivery_Band"], normalize="index").mul(100).round(2)
print("Band share by city (%):")
display(band_city)

,Orders
Delivery_Band,
Standard,817
Delayed,305
Fast,275
No Delivery,103


Band share by city (%):


Delivery_Band,Delayed,Fast,No Delivery,Standard
City,,,,
Bengaluru,17.79,19.76,7.11,55.34
Chennai,13.48,21.28,5.67,59.57
Delhi,22.80,10.75,6.51,59.93
Hyderabad,15.61,23.70,8.67,52.02
Kolkata,20.95,12.38,7.62,59.05
Meerut,3.64,54.55,8.18,33.64
Mumbai,36.27,3.05,6.78,53.90
Pune,9.48,33.62,4.31,52.59


## Task 14: Revenue by city and payment mode
Revenue counts only non-cancelled orders.

In [53]:
# Task 14
revenue = df[df["Order_Status"] != "Cancelled"]
rev_pivot = pd.pivot_table(revenue, values="Order_Value", index="City",
                           columns="Payment_Mode", aggfunc="sum",
                           margins=True, margins_name="Total").round(2)
display(rev_pivot)

Payment_Mode,Card,Cash,UPI,Wallet,Total
City,,,,,
Bengaluru,16650.56,15347.33,31847.49,7444.77,71290.15
Chennai,8025.38,7446.89,19516.31,5027.65,40016.23
Delhi,23401.72,12612.67,41092.99,10337.25,87444.63
Hyderabad,13897.21,7751.15,22046.78,5836.17,49531.31
Kolkata,5961.40,4192.97,15534.40,5196.02,30884.79
Meerut,7812.85,4037.07,14129.71,5957.48,31937.11
Mumbai,24187.44,14660.29,39649.67,6214.53,84711.93
Pune,6746.89,7041.80,15596.98,3300.02,32685.69
Total,106683.45,73090.17,199414.33,49313.89,428501.84


## Summary tables (expected deliverables)

In [54]:
# City performance table
city_perf = pd.concat([
    df.groupby("City").size().rename("Orders"),
    avg_city.rename("Avg Delivery (min)"),
    city_rates,
    revenue.groupby("City")["Order_Value"].sum().round(2).rename("Revenue"),
], axis=1).sort_values("Avg Delivery (min)")
print("CITY PERFORMANCE TABLE")
display(city_perf)

CITY PERFORMANCE TABLE


,Orders,Avg Delivery (min),Cancellation Rate (%),Delay Rate (%),Revenue
City,,,,,
Meerut,110,30.20,8.18,3.64,31937.11
Pune,116,34.56,4.31,9.48,32685.69
Chennai,141,36.42,5.67,14.18,40016.23
Hyderabad,173,36.83,8.67,15.61,49531.31
Bengaluru,253,38.08,7.11,18.58,71290.15
Kolkata,105,40.60,7.62,20.95,30884.79
Delhi,307,41.18,6.51,22.80,87444.63
Mumbai,295,43.60,6.78,36.95,84711.93


In [55]:
# Restaurant ranking
rest_rank = pd.concat([
    df.groupby("Restaurant")["Order_Value"].sum().round(2).rename("Total Value"),
    df.groupby("Restaurant")["Order_Value"].mean().round(2).rename("Avg Value"),
    df.groupby("Restaurant").size().rename("Orders"),
    (df.assign(d=df["Order_Status"].eq("Delayed")).groupby("Restaurant")["d"].mean() * 100).round(2).rename("Delay Rate (%)"),
], axis=1).sort_values("Total Value", ascending=False)
rest_rank.insert(0, "Rank", range(1, len(rest_rank) + 1))
print("RESTAURANT RANKING")
display(rest_rank)

RESTAURANT RANKING


,Rank,Total Value,Avg Value,Orders,Delay Rate (%)
Restaurant,,,,,
Pizza Corner,1,57511.11,463.80,124,25.00
Tandoori Nights,2,53427.16,414.16,129,20.93
Biryani House,3,45391.08,378.26,120,19.17
Spice Route,4,41819.25,316.81,132,20.45
Noodle Bar,5,40576.85,312.13,130,18.46
Dragon Wok,6,38201.47,318.35,120,23.33
Curry Leaf,7,32979.55,268.13,123,20.33
Burger Barn,8,32885.10,254.92,129,20.16
Dosa Plaza,9,29500.75,213.77,138,21.74


In [56]:
# Delay analysis
delay_analysis = pd.concat([
    df.groupby("Order_Hour")["Order_Status"].apply(lambda s: (s == "Delayed").mean() * 100).round(2).rename("Delay Rate (%)"),
    df.groupby("Order_Hour")["Delivery_Time_Min"].mean().round(2).rename("Avg Delivery (min)"),
], axis=1)
print("DELAY ANALYSIS BY HOUR")
display(delay_analysis)

DELAY ANALYSIS BY HOUR


,Delay Rate (%),Avg Delivery (min)
Order_Hour,,
10,8.11,34.23
11,18.00,39.63
12,12.28,37.30
13,37.04,43.76
14,7.27,33.94
15,30.56,39.43
16,10.26,34.83
17,16.09,35.80
18,13.77,36.55


## Task 15: Five operational recommendations
Generated from the results above so they always match the data.

In [57]:
# Task 15
slow_city = avg_city.idxmax(); fast_city = avg_city.idxmin()
worst_delay_city = city_rates["Delay Rate (%)"].idxmax()
worst_cancel_city = city_rates["Cancellation Rate (%)"].idxmax()
peak_list = ", ".join(f"{h}:00" for h in peak_hours.index.sort_values())
worst_hour = delay_analysis["Delay Rate (%)"].idxmax()
delay_rest = rest_rank["Delay Rate (%)"].idxmax()
top_rest = rest_rank.index[0]
top_pay = rev_pivot.loc["Total"].drop("Total").idxmax()

recs = [
    f"1. Fix slow cities: {slow_city} has the longest average delivery ({avg_city.max():.1f} min) vs {fast_city} ({avg_city.min():.1f} min). Add riders and review delivery zones in {slow_city}.",
    f"2. Staff for peak hours: orders peak at {peak_list}. Schedule extra riders and kitchen capacity in these slots; delays are highest around {worst_hour}:00 ({delay_analysis['Delay Rate (%)'].max():.1f}%).",
    f"3. Reduce delays in {worst_delay_city} ({city_rates['Delay Rate (%)'].max():.1f}% delayed) and audit {delay_rest}, the restaurant with the highest delay rate ({rest_rank['Delay Rate (%)'].max():.1f}%); set preparation-time targets with its kitchen.",
    f"4. Cut cancellations: {worst_cancel_city} has the highest cancellation rate ({city_rates['Cancellation Rate (%)'].max():.1f}%). Show accurate ETAs at checkout and send proactive delay alerts.",
    f"5. Protect top revenue sources: {top_rest} leads in total order value and {top_pay} is the biggest payment mode by revenue. Offer partner incentives to top restaurants and cashback on {top_pay} to keep growth.",
]
print("OPERATIONAL RECOMMENDATIONS\n")
for r in recs:
    print(r, "\n")

OPERATIONAL RECOMMENDATIONS

1. Fix slow cities: Mumbai has the longest average delivery (43.6 min) vs Meerut (30.2 min). Add riders and review delivery zones in Mumbai. 

2. Staff for peak hours: orders peak at 12:00, 19:00, 20:00. Schedule extra riders and kitchen capacity in these slots; delays are highest around 13:00 (37.0%). 

3. Reduce delays in Mumbai (37.0% delayed) and audit Pizza Corner, the restaurant with the highest delay rate (25.0%); set preparation-time targets with its kitchen. 

4. Cut cancellations: Hyderabad has the highest cancellation rate (8.7%). Show accurate ETAs at checkout and send proactive delay alerts. 

5. Protect top revenue sources: Pizza Corner leads in total order value and UPI is the biggest payment mode by revenue. Offer partner incentives to top restaurants and cashback on UPI to keep growth. 

